<a href="https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/28_ai_evals_offline_dataset/notebook_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 28: Creating Datasets for AI Evals

In this lesson, we'll explore how to create an evaluation dataset for Brown, the writing workflow.

**Learning Objectives:**

- Understand the structure and format of evaluation datasets for article generation
- Learn how to use the `EvalDataset` and `EvalSample` entities to load and manage evaluation data
- Upload evaluation datasets to Opik for tracking and analysis

> **Exercise version.** This is the exercise notebook for Lesson 28. The full solution lives in [`notebook.ipynb`](https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/28_ai_evals_offline_dataset/notebook.ipynb) in the same folder. Attempt each exercise before checking the solutions.

### Exercise roadmap

| Exercise | Difficulty | What you build |
|---|---|---|
| 1 | Starter | Loading and inspecting the dataset's metadata file |
| 2 | Starter | A directory walk over the evaluation samples |
| 3 | Intermediate | Loading the dataset into the `EvalDataset` entity |
| 4 | Starter | The upload step that pushes the dataset to Opik |

> [!NOTE]
> 💡 Remember that you can also run `brown` as a standalone Python package by going to `lessons/writing_workflow/` and following the instructions from there. We have a script at `lessons/writing_workflow/scripts/brown_create_eval_dataset.py` that you can use to upload datasets to Opik as well.

## 1. Setup


### Set Up Python Environment

**Google Colab:** Run the code cell below — it installs all required packages and loads your `OPIK_API_KEY` from Colab Secrets automatically.

To set up your Python virtual environment using `uv` and load it into the Notebook, follow the step-by-step instructions from the `Course Admin` lesson from the beginning of the course.

**TL/DR:** Be sure the correct kernel pointing to your `uv` virtual environment is selected.

In [ ]:
# ============================================================
# Google Colab Setup — runs only when executed in Colab
# ============================================================
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import importlib
    import os
    import site
    import subprocess

    # Install the course package (published from pyproject.toml)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "agentic-ai-engineering-course==0.4.8",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without restart

    # Load API key from Colab Secrets
    # In Colab: Secrets tab (key icon) → Add new secret → Name: OPIK_API_KEY
    from google.colab import userdata

    os.environ["OPIK_API_KEY"] = userdata.get("OPIK_API_KEY")

In [ ]:
if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

    from utils import env

    env.load(required_env_vars=["OPIK_API_KEY"])

### Import Key Packages

In [ ]:
import nest_asyncio
from utils import pretty_print

nest_asyncio.apply()  # Allow nested async usage in notebooks

### Download Required Files

First, let's download the configs folder:

In [ ]:
%%capture

!rm -rf configs
!curl -L -o configs.zip https://raw.githubusercontent.com/towardsai/agentic-ai-engineering-course/main/data/configs.zip
!unzip configs.zip
!rm -rf configs.zip

Now, we need to download the inputs folder containing the dataset files:

In [ ]:
%%capture

!rm -rf inputs
!curl -L -o inputs.zip https://raw.githubusercontent.com/towardsai/agentic-ai-engineering-course/main/data/inputs.zip
!unzip inputs.zip
!rm -rf inputs.zip

Let's verify what we downloaded:

In [ ]:
%ls

In [ ]:
from pathlib import Path

INPUTS_DIR = Path("inputs")

print(f"Inputs directory exists: {INPUTS_DIR.exists()}")

In [ ]:
EVALS_DATASET_DIR = Path("inputs/evals")

print(f"Examples directory exists: {EVALS_DATASET_DIR.exists()}")

## 2. Exploring The Evals Dataset Dir

### Exercise 1: Inspect the dataset metadata

An eval dataset starts with its manifest: `metadata.json` describes every sample before you touch any markdown file. Reading it first tells you what the loader will later expect.

**Learning goal:** Locate and inspect the metadata manifest that defines the evaluation dataset.

**What you need to implement:**

1. Build the path to `metadata.json`: it lives in the `dataset` subdirectory of `EVALS_DATASET_DIR`
2. Load the file's JSON content into `metadata`

**Key concepts:**

- `Path` objects compose with `/`, no string concatenation needed
- The manifest is a list of sample entries, each carrying a `name`, a `directory`, and optional per-file path overrides plus the `is_few_shot_example` flag

**Expected output:** the pretty-printed manifest: a JSON list of sample entries with names and directories.

**Implementation hints:**

- Look at the `EvalSample` fields documented further down to interpret what each metadata key feeds
- No API calls here, this is pure file inspection

In [ ]:
# === Exercise cell: fill in the gaps below ===

import json

# Steps to complete:
# 1. Build the path to metadata.json inside the dataset subdirectory of EVALS_DATASET_DIR
# 2. Load its JSON content into `metadata`

metadata_path = None  # TODO 1: the path to the manifest
metadata = None  # TODO 2: the loaded JSON content

# Your implementation goes here

# Smoke test
if metadata is not None:
    pretty_print.wrapped(json.dumps(metadata, indent=4), title="Evals Dataset Metadata")
else:
    print("(metadata not loaded yet, complete the TODOs above)")

Let's take a deeper look at our dataset, starting with it's overall structure:

In [ ]:
data_dir = EVALS_DATASET_DIR / "dataset" / "data"

pretty_print.wrapped(
    json.dumps(
        {
            "dataset_directory": str(EVALS_DATASET_DIR),
            "metadata_file": str(metadata_path),
            "data_directory": str(data_dir),
            "article_samples": len(list(data_dir.iterdir())),
        },
        indent=4,
    ),
    title="Evals Dataset Data Directory",
)

Now, let's look at each sample individually:

### Exercise 2: Walk the sample directories

The manifest names the samples, the filesystem holds them. Walking the data directory shows what files each sample actually ships: guideline, research, and ground-truth article.

**Learning goal:** Explore the on-disk layout of evaluation samples.

**What you need to implement:**

1. Loop over the entries of `data_dir` in sorted order, keeping only directories
2. For each sample directory: print its name (with a trailing `/`), collect the names of the regular files inside, and print each file indented on its own line
3. Append each sample directory's name to `sample_dirs_found`

**Key concepts:**

- `Path.iterdir()` yields entries in arbitrary order, `sorted()` makes the output stable
- `.is_dir()` and `.is_file()` filter out strays

**Expected output:** one block per article sample: the folder name followed by its markdown files, e.g. an `article_guideline.md`, a `research.md`, and a ground-truth article file.

**Implementation hints:**

- This is the same structure `EvalDataset.load_dataset` will read programmatically in Exercise 3, keep the file names in mind
- Compare what you see against the metadata from Exercise 1: does every manifest entry have a folder?

In [ ]:
# === Exercise cell: fill in the gaps below ===

pretty_print.wrapped("ARTICLE SAMPLES", indent=38)

# Steps to complete:
# 1. Loop over data_dir's entries in sorted order, keeping only directories
# 2. Print each sample directory's name, then its files indented one per line
# 3. Append each directory's name to sample_dirs_found

sample_dirs_found = []  # TODO: fill while looping

# Your implementation goes here

# Smoke test
print(f"{len(sample_dirs_found)} sample directories listed" if sample_dirs_found else "(nothing listed yet, complete the loop above)")

## 3. Uploading The Evals Dataset To Opik

We will quickly go over the code used to upload the dataset described above to Opik. The code is pretty minimal. Thus, we will keep it short.

### 3.1 The EvalSample Entity

The `EvalSample` entity is a Pydantic model that represents a single evaluation sample containing all the data needed for article generation evaluation.

Source: `brown.evals.dataset`
```python
class EvalSample(BaseModel):
    name: str
    directory: Path
    article_guideline: str
    research: str
    ground_truth_article: str
    is_few_shot_example: bool = False
```

Each sample contains:
- `name`: A human-readable identifier for the sample
- `directory`: The path where the sample files are located
- `article_guideline`: The writing guidelines in markdown format
- `research`: The research/source material in markdown format
- `ground_truth_article`: The reference article to compare against
- `is_few_shot_example`: Whether this sample is used for few-shot learning instead of evaluation

### 3.2 The EvalDataset Entity

The `EvalDataset` entity is a Pydantic model that represents a collection of evaluation samples along with dataset metadata.

Source: `brown.evals.dataset`
```python
class EvalDataset(BaseModel):
    name: str
    description: str
    samples: list[EvalSample]

    @classmethod
    def load_dataset(cls, directory: Path, name: str, description: str) -> Self:
        metadata_file = directory / "metadata.json"
        if not metadata_file.exists():
            raise FileNotFoundError(f"Metadata file not found: {metadata_file}")

        with metadata_file.open() as f:
            metadata = json.load(f)

        samples = []
        for sample_metadata in metadata:
            sample_dir = directory / sample_metadata["directory"]

            article_guideline = cls._load_markdown_file(
                sample_dir / sample_metadata.get("article_guideline_path", DEFAULT_ARTICLE_GUIDELINE_PATH)
            )
            research = cls._load_markdown_file(sample_dir / sample_metadata.get("research_path", DEFAULT_RESEARCH_PATH))
            ground_truth_article = cls._load_markdown_file(
                sample_dir / sample_metadata.get("ground_truth_article_path", DEFAULT_GROUND_TRUTH_ARTICLE_PATH)
            )

            sample = EvalSample(
                name=sample_metadata["name"],
                directory=sample_metadata["directory"],
                is_few_shot_example=sample_metadata.get("is_few_shot_example", False),
                article_guideline=article_guideline,
                research=research,
                ground_truth_article=ground_truth_article,
            )
            samples.append(sample)

        return cls(name=name, description=description, samples=samples)
```

The `load_dataset` class method:
- Reads the `metadata.json` file from the specified directory
- Iterates through each sample entry and loads the corresponding markdown files
- Creates `EvalSample` instances for each entry
- Returns a fully populated `EvalDataset` ready for use


### 3.3 The upload_dataset Function

The `upload_dataset` function uploads the evaluation dataset to the Opik observability platform for tracking and analysis.

Source: `brown.observability.dataset`
```python
def upload_dataset(evaluation_dataset: "EvalDataset") -> None:
    samples = evaluation_dataset.model_dump(mode="json")["samples"]
    eval_samples = [sample for sample in samples if not sample["is_few_shot_example"]]
    logger.info(f"Uploading `{len(eval_samples)}/{len(samples)}` evaluation samples to Opik.")
    training_samples = [sample for sample in samples if sample["is_few_shot_example"]]
    logger.info(f"The following `{len(training_samples)}/{len(samples)}` samples will be used for training or as few-shot examples:")
    for sample in training_samples:
        logger.info(f"- `{sample['name']}`")

    opik_utils.update_or_create_dataset(
        name=evaluation_dataset.name,
        description=evaluation_dataset.description,
        items=eval_samples,
    )
```

The function:
- Separates samples into evaluation samples and few-shot examples based on the `is_few_shot_example` flag
- Only uploads evaluation samples to Opik (few-shot examples are used by the LLM judge. Thus, to avoid data leakage, we cannot compute metrics on them)

While the `update_or_create_dataset` function handles updating the dataset on Opik.

Source: `brown.observability.opik_utils`
```python
import opik

def update_or_create_dataset(name: str, description: str, items: list[dict]) -> opik.Dataset:
    """
    Update an existing dataset or create a new one if it doesn't exist.

    Args:
        name: The name of the dataset to update or create.
        description: The description of the dataset.
        items: The items to insert into the dataset.

    Returns:
        opik.Dataset: The updated or created dataset.
    """

    client = opik.Opik()
    dataset = client.get_or_create_dataset(name=name, description=description)
    dataset.clear()

    dataset.insert(items)

    return dataset
```

This is a simple function that gets or creates a dataset on Opik based on its name. Then it clears the dataset and reinserts all the items. As our dataset is small, doing this reinsertion every time works fine, making it a good strategy to avoid duplicates.

### 3.4 Loading and Uploading the Dataset

Now let's use the `EvalDataset` entity to load our evaluation dataset and upload it to Opik. First, let's reference our dataset directory:


In [ ]:
EVALS_DATASET_DIR

In [ ]:
INPUT_EVALS_DATASET_DIR = EVALS_DATASET_DIR / "dataset"
DATASET_NAME = "brown-course-lessons"
DATASET_DESCRIPTION = "Brown evaluation dataset on course lessons format."

Now, let's load the dataset:

### Exercise 3: Load the dataset with the EvalDataset entity

Everything you inspected by hand, the `EvalDataset` entity loads in one call: it reads the manifest, pulls each sample's markdown files, and validates the lot into typed `EvalSample` objects.

**Learning goal:** Load an evaluation dataset into its typed entity and inspect the result.

**What you need to implement:**

1. Call `EvalDataset.load_dataset` with `INPUT_EVALS_DATASET_DIR` plus the `DATASET_NAME` and `DATASET_DESCRIPTION` constants, storing the result in `dataset`

**Key concepts:**

- `load_dataset` is a classmethod: directory first, then `name` and `description` keyword arguments
- The returned entity carries `name`, `description`, and a `samples` list of `EvalSample` objects, the quoted source above shows exactly how it assembles them

**Expected output:** a "Dataset Metadata" block reporting the dataset name, description, and its sample count.

**Implementation hints:**

- Point it at the `dataset` subdirectory (that is what `INPUT_EVALS_DATASET_DIR` already holds), not the outer evals folder
- The upload cell below consumes `dataset`, it stays blocked until this loads

In [ ]:
# === Exercise cell: fill in the gaps below ===

from brown.evals.dataset import EvalDataset
from brown.observability import upload_dataset
from loguru import logger

# Step to complete: load the dataset from INPUT_EVALS_DATASET_DIR with the
# DATASET_NAME and DATASET_DESCRIPTION constants

dataset = None  # TODO: load the dataset entity

# Your implementation goes here

# Smoke test
if dataset is not None:
    pretty_print.wrapped(
        {
            "dataset_name": dataset.name,
            "dataset_description": dataset.description,
            "dataset_samples": len(dataset.samples),
        },
        title="Dataset Metadata",
    )
else:
    print("(dataset not loaded yet, complete the TODO above)")

### Validation check - run this after your implementation

Uncomment the cell below and run it once Exercise 3 has loaded the dataset. Local checks only, no Opik calls.

In [ ]:
# # Validation: check your Exercise 3 implementation
# try:
#     assert dataset is not None, "❌ 'dataset' is not set. Run your Exercise 3 cell first."
#     assert dataset.name == DATASET_NAME, f"❌ Expected the dataset name {DATASET_NAME!r}."
#     assert dataset.samples, "❌ The dataset has no samples. Check the directory you loaded from."
#     _s = dataset.samples[0]
#     assert _s.article_guideline.strip(), "❌ Sample article_guideline is empty, the markdown files did not load."
#     assert _s.research.strip() and _s.ground_truth_article.strip(), "❌ Sample research or ground-truth article is empty."
#     _few_shot = sum(1 for s in dataset.samples if s.is_few_shot_example)
#     print(f"✅ All checks passed! {len(dataset.samples)} samples loaded ({_few_shot} few-shot).")
# except AssertionError as e:
#     print(e)
#     print("💡 Tip: load from the dataset subdirectory and pass name/description as keyword arguments.")

Finally, let's upload the dataset to Opik:


### Exercise 4: Upload the dataset to Opik

The final step publishes the dataset to Opik so every future evaluation run scores against the same tracked samples. The `upload_dataset` function (quoted above) handles the few-shot split for you.

**Learning goal:** Push a loaded evaluation dataset to the observability platform.

**What you need to implement:**

1. Log which dataset is being uploaded (its name)
2. Upload the loaded `dataset` with `upload_dataset`
3. Log a success message and set `uploaded` to `True`

**Key concepts:**

- Few-shot samples are excluded from the upload automatically, they feed the LLM judge and would leak if scored
- The underlying Opik call clears and reinserts, so re-running never duplicates items

**Expected output:** loguru lines reporting the upload of the evaluation samples (few-shot samples listed as excluded) and your success message, then the dataset appears in your Opik workspace.

**Implementation hints:**

- `logger.info` before, `logger.success` after, with the dataset name interpolated
- This makes real Opik API calls, complete Exercise 3 first so `dataset` is a loaded entity

In [ ]:
# === Exercise cell: fill in the gaps below ===

# Steps to complete:
# 1. Log which dataset is being uploaded
# 2. Upload the loaded dataset to Opik with upload_dataset
# 3. Log a success message and set uploaded = True

uploaded = False

# Your implementation goes here

# Smoke test
if not uploaded:
    print("(not uploaded yet, complete the steps above)")

## 4. Conclusion

In this lesson, we learned how to create and manage evaluation datasets for the Brown writing workflow.

In the next lesson, we'll use this dataset to run evaluations and measure the quality of Brown.

### Practicing Ideas

1. Extend the dataset with more diverse article samples.
2. Change the dataset with a new set of articles that follow your format instead of our course lesson format.
3. Do your own AI evals dataset on a different data format, such as social media posts.
4. Update the `update_or_create_dataset` to stop `clearing` the dataset entirely when adding new items by introducing a mechanism to detect dataset item duplicates.
5. Use Opik to version the dataset when changing it in any way, such as adding, removing or changing dataset samples.

> [!NOTE]
> 💡 Remember that you can also run `brown` as a standalone Python package by going to `lessons/writing_workflow/` and following the instructions from there. We have a script at `lessons/writing_workflow/scripts/brown_create_eval_dataset.py` that you can use to upload datasets to Opik as well.
